In [ ]:
import sys
sys.path.append('..')  # Go up one level
import matching_decoupling_network as network
import numpy as np
import pandas

# A Four-Element Matching (Impedance Transform) Network Generator

M, N: TI shape ((c) below)

O, P: Π¯ shape ((d) below)

Component numbering:
Coil ← 1 2 3 4 → Amplifier

<img src="../assets/jn_matching_networks_4.png" alt="Three-Element Network" width="360" />

Need: 
- Frequency in hertz
- $Z_\mathrm{amp}$: in ohm
- $Z_\mathrm{out}$: in ohm
- $Z_\mathrm{coil}$: in ohm

The quantities above are defined in the picture below. 

<img src="../assets/preamplifier_decoupling_configuration.png" alt="Preamplifier Decoupling Configuration" width="600" />

Cases can be "HZ" for high impedance, "LZ" for low impedance: 
- To suppress the loop current of a loop MRI coil (i.e. an electrically small loop antenna), use "HZ". In this case:
    - Set $Z_\mathrm{coil}$ as the coil impedance. A rough coil reactance and a rough estimation of the coil resistance will usually make a decent initial design. 
    - Set $Z_\mathrm{out}$ as the optimal noise impedance of the preamplifier module. 
    - Set $Z_\mathrm{amp}$ as the input impedance of the preamplifier module.
    - $Z_\mathrm{in}$ will be automatically calculated.
- To make low-impedance amplifiers, use "LZ". In this case: 
    - Set $Z_\mathrm{coil}$ as the target optimal noise impedance, e.g. $Z_\mathrm{coil}=50~\Omega$, or anything you need.
    - Set $Z_\mathrm{out}$ as the optimal noise impedance of the "true" amplifier, i.e. the optimal noise impedance seen at the input of the very first transistor stage.
    - Set $Z_\mathrm{amp}$ as the input impedance of the "true" amplifier, i.e. the input impedance seen at the input of the very first transistor stage.
    - $Z_\mathrm{in}$ will be automatically calculated
- To form preamp decoupling for parallel self-resonant coil loops, use "LZ". In this case:
    - Set $Z_\mathrm{coil}$ as the coil impedance. A rough coil reactance and a rough estimation of the coil resistance will usually make a decent initial design. 
    - Set $Z_\mathrm{out}$ as the optimal noise impedance of the preamplifier module.
    - Set $Z_\mathrm{amp}$ as the input impedance of the preamplifier module.
    - $Z_\mathrm{in}$ will be automatically calculated.


## Example: Four-Element Matching Network for a Loop MRI Coil

Here we generate a four-element matching (impedance transform) network for a loop MRI coil. We use the amplifier data in case (C) in Table 1 of \[1\], namely, 
```math
\begin{aligned}
f &= 32.125~\mathrm{MHz} \\
Z_\mathrm{amp} &= 583 \angle -83.6^\circ~\Omega \\
Z_\mathrm{out} &= 114 \angle 10.2^\circ~\Omega \\
Z_\mathrm{coil} &= 0.382 + \mathrm{j} 33.9~\Omega 
\end{aligned}
```

We choose the case "HZ". 

Unlike three-element networks which have only four solutions, for four-element networks, there are infinitely many. The fourth element is an extra degree of freedom with which the designer can generate a list of networks and choose one (s)he deem the fittest. It will be most convenient to filter out the networks with unrealistic component values. Besides, when a network contains too many inductors, the SNR impairment of the matching network will likely be high, especially for electrically very small antennae (very small loops) operating at a low frequency. So we need to filter out those "unsuitable" networks.

In this example, 
- We use $1~\mathrm{pF} \leq C \leq 1000~\mathrm{pF}$ capacitors and $1~\mathrm{nH} \leq L \leq 709~\mathrm{nH}$ for the 4th element. The steps are 2 pF or 2 nH.
- We limit the capacitance range to $0.1~\mathrm{pF} \leq C \leq 2000~\mathrm{pF}$ for all capacitors. This is set in `c_min_F` and `c_max_F`.
- We limit the inductance range to $1~\mathrm{nH} \leq L \leq 709~\mathrm{nH}$ for all inductors. This is set in `l_min_H` and `l_max_H`.
- We limit the number of inductors to 1. That is, a network can contain at most 1 inductor. This is set in `inductor_limit`.

There is an extra parameter `pandas.options.display.max_rows`. It controls in how many rows a dataframe can be printed out. 

↓---The following parameters can be changed---↓

In [ ]:
freq = 32.125e6   # Frequency, Hz
z_amp = 583. * np.exp(1j * np.deg2rad(-83.6))  # Amplifier input impedance, Ohm
z_out = 114. * np.exp(1j * np.deg2rad(10.2))  # The impedance to transfer the coil Z to, Ohm. It's often the noise optimum.
z_coil = 0.382 + 33.9j   # Coil impedance, Ohm
category = "HZ"   # Can be "HZ" or "LZ"

iv_c_values = np.arange(1, 1000, 2) * 1e-12   # Range of the fourth (nearest to the amplifier) capacitor, F.
iv_l_values = np.arange(1, 709, 2) * 1e-9     # Range of the fourth (nearest to the amplifier) inductor, H.
c_min_F, c_max_F = 0.1e-12, 2000e-12  # Minimum and maximum capacitance, F.
l_min_H, l_max_H = 1e-9,  709e-9  # Minimum and maximum inductance, H.
inductor_limit = 1  # Number of inductors acceptable

pandas.options.display.max_rows = 4096  # Display at maximum 4000 entries

↑---The above parameters can be changed---↑

### Calculation

In [ ]:
decoupling = network.max_preamplifier_decoupling(z_out, z_amp)
relative_gain = network.gain_relative_to_maximum_available(z_out, z_amp)

# Convert the 4th element to reactances
iv_values = np.hstack((iv_c_values, iv_l_values))
iv_units = [*(["F"] * np.size(iv_c_values)), *(["H"] * np.size(iv_l_values))]
x_iv = np.hstack((-1 / (2 * np.pi * freq * iv_c_values), 2 * np.pi * freq * iv_l_values))
z_amp_tilde_pi = z_amp + x_iv * 1j
z_out_tilde_pi = z_out - x_iv * 1j
z_amp_tilde_t = 1 / (1 / z_amp + 1 / (1j * x_iv))
z_out_tilde_t = 1 / (1 / z_out - 1 / (1j * x_iv))

# Do some transform as illustrated in (c), (d) above.
match category:
    case "HZ":
        x11_tilde_pi, xø_p_tilde_pi, x22_tilde_pi = network.impedance_matrix_hz(z_coil, z_amp_tilde_pi, z_out_tilde_pi)
        x11_tilde_t, xø_p_tilde_t, x22_tilde_t =    network.impedance_matrix_hz(z_coil, z_amp_tilde_t,  z_out_tilde_t)
        x11, xø_p, x22 = network.impedance_matrix_hz(z_coil, z_amp, z_out)
        z_pr = network.power_swr(z_out, z_amp) * np.real(z_coil) - 1j * np.imag(z_coil)  # input impedance Z_{in}
    case "LZ":
        x11_tilde_pi, xø_p_tilde_pi, x22_tilde_pi = network.impedance_matrix_lz(z_coil, z_amp_tilde_pi, z_out_tilde_pi)
        x11_tilde_t, xø_p_tilde_t, x22_tilde_t =    network.impedance_matrix_lz(z_coil, z_amp_tilde_t,  z_out_tilde_t)
        x11, xø_p, x22 = network.impedance_matrix_lz(z_coil, z_amp, z_out)
        z_pr = np.real(z_coil) / network.power_swr(z_out, z_amp) - 1j * np.imag(z_coil)
        
xM1_tilde_t, xM2_tilde_t, xM3_tilde_t = network.reactance_matrix_to_T(x11_tilde_t, xø_p_tilde_t, x22_tilde_t)
xN1_tilde_t, xN2_tilde_t, xN3_tilde_t = network.reactance_matrix_to_T(x11_tilde_t, -xø_p_tilde_t, x22_tilde_t)
yO1_tilde_pi, yO2_tilde_pi, yO3_tilde_pi = network.reactance_matrix_to_Pi(x11_tilde_pi, xø_p_tilde_pi, x22_tilde_pi)
yP1_tilde_pi, yP2_tilde_pi, yP3_tilde_pi = network.reactance_matrix_to_Pi(x11_tilde_pi, -xø_p_tilde_pi, x22_tilde_pi)

# Convert reactance and susceptance to corresponding L, C components
M1_to_N3_tuple = tuple(map(lambda x: network.reactance_to_LC(x, freq),
                           [xM1_tilde_t, xM2_tilde_t, xM3_tilde_t, xN1_tilde_t, xN2_tilde_t, xN3_tilde_t]))
O1_to_P3_tuple = tuple(map(lambda x: network.susceptance_to_LC(x, freq),
                           [yO1_tilde_pi, yO2_tilde_pi, yO3_tilde_pi, yP1_tilde_pi, yP2_tilde_pi, yP3_tilde_pi]))

# Gather into networks
t_network_M = {
    "E1": M1_to_N3_tuple[0][0], "uE1": M1_to_N3_tuple[0][1],
    "E2": M1_to_N3_tuple[1][0], "uE2": M1_to_N3_tuple[1][1],
    "E3": M1_to_N3_tuple[2][0], "uE3": M1_to_N3_tuple[2][1],
    "E4": iv_values,         "uE4": iv_units,
}
t_network_N = {
    "E1": M1_to_N3_tuple[3][0], "uE1": M1_to_N3_tuple[3][1],
    "E2": M1_to_N3_tuple[4][0], "uE2": M1_to_N3_tuple[4][1],
    "E3": M1_to_N3_tuple[5][0], "uE3": M1_to_N3_tuple[5][1],
    "E4": iv_values,         "uE4": iv_units
}
pi_network_O = {
    "E1": O1_to_P3_tuple[0][0], "uE1": O1_to_P3_tuple[0][1],
    "E2": O1_to_P3_tuple[1][0], "uE2": O1_to_P3_tuple[1][1],
    "E3": O1_to_P3_tuple[2][0], "uE3": O1_to_P3_tuple[2][1],
    "E4": iv_values,         "uE4": iv_units,
}
pi_network_P = {
    "E1": O1_to_P3_tuple[3][0], "uE1": O1_to_P3_tuple[3][1],
    "E2": O1_to_P3_tuple[4][0], "uE2": O1_to_P3_tuple[4][1],
    "E3": O1_to_P3_tuple[5][0], "uE3": O1_to_P3_tuple[5][1],
    "E4": iv_values,         "uE4": iv_units
}
pandas.set_option('display.max_rows', None)
t_network_M_dtf = pandas.DataFrame(t_network_M)
t_network_N_dtf = pandas.DataFrame(t_network_N)
pi_network_O_dtf = pandas.DataFrame(pi_network_O)
pi_network_P_dtf = pandas.DataFrame(pi_network_P)

# Filter out networks with too many inductors
t_network_M_dtf = network.filter_out_networks_of_too_many_inductors(t_network_M_dtf, np.int32(inductor_limit))
t_network_N_dtf = network.filter_out_networks_of_too_many_inductors(t_network_N_dtf, np.int32(inductor_limit))
pi_network_O_dtf = network.filter_out_networks_of_too_many_inductors(pi_network_O_dtf, np.int32(inductor_limit))
pi_network_P_dtf = network.filter_out_networks_of_too_many_inductors(pi_network_P_dtf, np.int32(inductor_limit))

# Filter out networks with out-of-range components
t_network_M_dtf = network.threshold_component_values(t_network_M_dtf, l_min_H, l_max_H, c_min_F, c_max_F)
t_network_N_dtf = network.threshold_component_values(t_network_N_dtf, l_min_H, l_max_H, c_min_F, c_max_F)
pi_network_O_dtf = network.threshold_component_values(pi_network_O_dtf, l_min_H, l_max_H, c_min_F, c_max_F)
pi_network_P_dtf = network.threshold_component_values(pi_network_P_dtf, l_min_H, l_max_H, c_min_F, c_max_F)


### Results

Preamplifier decoupling, gain, $Z_\mathrm{in}$, etc.

In [ ]:
print(f"Maximum preamplifier decoupling = {decoupling: .5f} = {-20 * np.log10(decoupling): .2f} dB")
print(f"Gain relative to the maximum available = {relative_gain: .5f} = {10 * np.log10(relative_gain): .2f} dB")
print(f"Z_in = {z_pr: .6g} Ω")
print(f"""X_11 = {x11: .6g} Ω
X_⌀ = ±{xø_p: .6g} Ω
X_22 = {x22: .6g} Ω
""")

#### Network M, TI

Component numbering:
Coil ← 1 2 3 4 → Amplifier

If you see "empty dataframe", it means the matching networks are all filtered out in that configuration because of having out-of-range components or having too many inductors.

In [ ]:
print("network M, TI------------\n", t_network_M_dtf)

#### Network N, TI

Component numbering:
Coil ← 1 2 3 4 → Amplifier

If you see "empty dataframe", it means the matching networks are all filtered out in that configuration because of having out-of-range components or having too many inductors.

In [ ]:
print("network N, TI------------\n", t_network_N_dtf)

#### Network O, Π¯------------

Component numbering:
Coil ← 1 2 3 4 → Amplifier

If you see "empty dataframe", it means the matching networks are all filtered out in that configuration because of having out-of-range components or having too many inductors.

In [ ]:
print("network O, Π¯------------\n", pi_network_O_dtf)

#### Network P, Π¯------------

Component numbering:
Coil ← 1 2 3 4 → Amplifier

If you see "empty dataframe", it means the matching networks are all filtered out in that configuration because of having out-of-range components or having too many inductors.

In [ ]:
print("network P, Π¯------------\n", pi_network_P_dtf)